In [5]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\gnfer\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
# Importar bibliotecas

import scipy as sp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix
from sklearn.utils import resample

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler 

In [7]:
#Import the data
df_eda = pd.read_csv("combined_Lisboa_porto_complete.csv")
df_eda_untouched = df_eda.copy()

# Em vez de estarem na mesma pasta do notebook, podem usar o caminho absoluto do ficheiro

# Porquê .copy()? Porque queremos um objeto independente, não um apontador para df_titanic
# Assim, cria-se uma cópia independente de um objeto. Evita-se que alterações na cópia afetem o objeto original

# Ex.:
# >>> a = [1, 2]  # cria um objeto na memória que aponta para 1 e 2, e aponta a para ele
# >>> b = a       # aponta b para onde a aponta (porque a é um objeto mutável)
# >>> a[0] = 2    # altera o objeto para onde a aponta, direcionando o seu primeiro item para 2
# >>> a
# [2, 2]
# >>> b
# [2, 2]

In [8]:
# Check data structure
df_eda.head()

,city,datetime,CO,NO2,O3,PM10,PM2.5,SO2,temperature_c,humidity_percent,pressure_hpa,wind_speed_kmh,wind_direction_deg,precipitation_mm,label_air_good
0,Lisboa,2025-09-05 01:00:00,0.96,25.20,84.09,11.82,9.12,6.75,18.9,82.0,1018.5,16.2,357.0,0.0,Yes
1,Lisboa,2025-09-05 02:00:00,0.75,26.40,86.20,13.24,8.87,5.11,18.8,80.0,1018.3,15.5,356.0,0.0,Yes
2,Lisboa,2025-09-05 03:00:00,0.87,25.16,74.41,15.18,10.84,5.76,18.6,79.0,1017.4,11.5,2.0,0.0,Yes
3,Lisboa,2025-09-05 04:00:00,0.51,13.59,68.57,17.48,13.14,5.03,18.3,77.0,1017.2,11.3,351.0,0.0,No
4,Lisboa,2025-09-05 05:00:00,0.61,15.89,78.79,14.70,13.68,6.20,18.6,72.0,1016.8,9.4,356.0,0.0,Yes


In [9]:
# Verificar os tipos de dados e obter mais informações
df_eda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1442 entries, 0 to 1441
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   city                1442 non-null   object 
 1   datetime            1442 non-null   object 
 2   CO                  1442 non-null   float64
 3   NO2                 1442 non-null   float64
 4   O3                  1442 non-null   float64
 5   PM10                1442 non-null   float64
 6   PM2.5               1442 non-null   float64
 7   SO2                 1442 non-null   float64
 8   temperature_c       1442 non-null   float64
 9   humidity_percent    1442 non-null   float64
 10  pressure_hpa        1442 non-null   float64
 11  wind_speed_kmh      1442 non-null   float64
 12  wind_direction_deg  1442 non-null   float64
 13  precipitation_mm    1442 non-null   float64
 14  label_air_good      1442 non-null   object 
dtypes: float64(12), object(3)
memory usage: 169.1+ KB


In [10]:
# Looking for abnormal outliers
df_eda.describe()

,CO,NO2,O3,PM10,PM2.5,SO2,temperature_c,humidity_percent,pressure_hpa,wind_speed_kmh,wind_direction_deg,precipitation_mm
count,1442.000000,1442.000000,1442.000000,1442.000000,1442.000000,1442.000000,1442.000000,1442.000000,1442.000000,1442.000000,1442.000000,1442.000000
mean,1.006546,29.701477,69.301706,21.614896,15.138800,6.453259,19.428502,69.087379,1019.587864,8.953051,234.529820,0.043967
std,0.442828,13.279275,16.780429,9.428184,6.889596,3.050010,3.778562,19.718614,3.062214,6.143344,122.062848,0.574530
min,0.120000,2.420000,28.220000,1.840000,0.100000,0.100000,10.000000,16.000000,1002.900000,0.000000,1.000000,0.000000
25%,0.650000,19.040000,56.552500,13.952500,9.650000,4.122500,17.000000,54.000000,1018.200000,3.700000,104.000000,0.000000
50%,0.940000,28.155000,69.205000,20.110000,14.080000,6.010000,19.200000,72.000000,1020.100000,7.300000,293.000000,0.000000
75%,1.300000,38.527500,81.310000,28.020000,19.940000,8.397500,21.700000,86.000000,1021.600000,13.000000,340.000000,0.000000
max,2.560000,76.470000,114.680000,58.570000,38.750000,18.360000,32.000000,100.000000,1025.700000,28.000000,360.000000,15.600000


In [15]:
# E as variáveis não numéricas, temos alguma informação sobre elas?

print('City Unique Values:', df_eda['city'].unique())
print('datetime Unique Values:', df_eda['datetime'].unique())
print('label_air_good Unique Values:', df_eda['label_air_good'].unique())
df_eda.describe(include=['O']) # estatísticas para variáveis do tipo string

City Unique Values: ['Lisboa' 'Porto']
datetime Unique Values: ['2025-09-05 01:00:00' '2025-09-05 02:00:00' '2025-09-05 03:00:00'
 '2025-09-05 04:00:00' '2025-09-05 05:00:00' '2025-09-05 06:00:00'
 '2025-09-05 07:00:00' '2025-09-05 08:00:00' '2025-09-05 09:00:00'
 '2025-09-05 10:00:00' '2025-09-05 11:00:00' '2025-09-05 12:00:00'
 '2025-09-05 13:00:00' '2025-09-05 14:00:00' '2025-09-05 15:00:00'
 '2025-09-05 16:00:00' '2025-09-05 17:00:00' '2025-09-05 18:00:00'
 '2025-09-05 19:00:00' '2025-09-05 20:00:00' '2025-09-05 21:00:00'
 '2025-09-05 22:00:00' '2025-09-05 23:00:00' '2025-09-06 00:00:00'
 '2025-09-06 01:00:00' '2025-09-06 02:00:00' '2025-09-06 03:00:00'
 '2025-09-06 04:00:00' '2025-09-06 05:00:00' '2025-09-06 06:00:00'
 '2025-09-06 07:00:00' '2025-09-06 08:00:00' '2025-09-06 09:00:00'
 '2025-09-06 10:00:00' '2025-09-06 11:00:00' '2025-09-06 12:00:00'
 '2025-09-06 13:00:00' '2025-09-06 14:00:00' '2025-09-06 15:00:00'
 '2025-09-06 16:00:00' '2025-09-06 17:00:00' '2025-09-06 18:00:00'

,city,datetime,label_air_good
count,1442,1442,1442
unique,2,721,2
top,Lisboa,2025-09-05 01:00:00,Yes
freq,721,2,958


In [16]:
df_eda.columns

Index(['city', 'datetime', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2',
       'temperature_c', 'humidity_percent', 'pressure_hpa', 'wind_speed_kmh',
       'wind_direction_deg', 'precipitation_mm', 'label_air_good'],
      dtype='object')

In [17]:
# Vamos verificar valores em falta!
df_eda.isnull().sum()

city                  0
datetime              0
CO                    0
NO2                   0
O3                    0
PM10                  0
PM2.5                 0
SO2                   0
temperature_c         0
humidity_percent      0
pressure_hpa          0
wind_speed_kmh        0
wind_direction_deg    0
precipitation_mm      0
label_air_good        0
dtype: int64